**张量（Tensor）** 是一种与数组和矩阵非常相似的专用数据结构。 在PyTorch中，我们使用张量编码模型的输入和输出，以及模型参数。

张量类似于NumPy的ndarray，不同之处在于张量可以在GPU或其他硬件加速器上运行。实际上，张量和NumPy数组通常可以共享相同的底层内存，从而消除复制数据的需求（参见与 NumPy 的相互转化）。张量也为自动微分优化（我们稍后会在Autograd部分详细介绍）。如果你熟悉ndarrays，Tensor API你会非常熟悉。如果没有，就跟着看吧！

In [1]:
import torch
import numpy as np

### PART1. 张量的初始化

张量可以从直接的数据中初始化：

In [2]:
data = [[1, 2],[3, 4]]
x_data = torch.tensor(data)

也可从numpy数组初始化：（实际上，二者可以相互转换）

In [3]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)

基于已有的张量，我们可以创建新的张量：

In [4]:
x_ones = torch.ones_like(x_data) # 这将返回一个与x_data形状相同的张量，并填充1
print(f"填充1的张量: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float) # 这里生成一个随机填充的张量，dtype可以指定张量的数据类型
print(f"随机填充的张量: \n {x_rand} \n")

填充1的张量: 
 tensor([[1, 1],
        [1, 1]]) 

随机填充的张量: 
 tensor([[0.8934, 0.1401],
        [0.0803, 0.5903]]) 



下方中shape是决定张量维数的元组（行，列）。

In [5]:
shape = (2,3)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor}")

Random Tensor: 
 tensor([[0.6272, 0.6547, 0.7601],
        [0.3706, 0.7999, 0.0561]]) 

Ones Tensor: 
 tensor([[1., 1., 1.],
        [1., 1., 1.]]) 

Zeros Tensor: 
 tensor([[0., 0., 0.],
        [0., 0., 0.]])


### PART2. 张量的属性

张量的属性包括它们的形状、数据类型以及存储张量的设备等：

In [6]:
tensor = torch.rand(3,4)

print(f"张量形状: {tensor.shape}")
print(f"张量的数据类型: {tensor.dtype}")
print(f"张量存储设备: {tensor.device}")

张量形状: torch.Size([3, 4])
张量的数据类型: torch.float32
张量存储设备: cpu


### PART3. 张量的运算

此处详尽介绍了1200 余种张量运算，涵盖算术运算、线性代数、矩阵操作（转置、索引、切片）、采样等各类操作。

上述全部运算均可在 CPU 以及各类加速硬件上运行，例如 CUDA、MPS、MTIA、XPU。如果你使用Colab环境（谷歌提供的云端 Jupyter Notebook），可以通过「Runtime > Change runtime type > GPU」来启用硬件加速。

张量默认在 CPU 上创建。我们需要先确认加速设备可用，再通过 .to() 方法，把张量显式迁移到加速设备。请记住：在不同设备之间拷贝大张量，会消耗大量时间与显存或内存资源！

In [7]:
# 如果当前存在可用的加速器，我们就将张量迁移至该加速器上。
if torch.accelerator.is_available():
    tensor = tensor.to(torch.accelerator.current_accelerator())

接下来展示一些对张量的操作（运算）：

**张量索引和切片**    

","用于隔开不同维度，":"表示该维度全部元素，"..."表示剩下所有维度（可用于高维张量）。

In [8]:
tensor = torch.ones(4, 4)
print(f"第一行: {tensor[0]}")
print(f"第一列: {tensor[:, 0]}")
print(f"最后一列: {tensor[..., -1]}")
tensor[:,1] = 0
print(tensor)

第一行: tensor([1., 1., 1., 1.])
第一列: tensor([1., 1., 1., 1.])
最后一列: tensor([1., 1., 1., 1.])
tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])


**张量拼接：** 你可以使用 shturl.cc 在指定维度上拼接一组张量。torch.stack是另一种张量拼接运算符，区别如下：

shturl.cc：不增加新维度，在现有的某一维上把张量拼在一起。例：两个[2,3]张量 cat → 结果还是二维，如[4,3]

torch.stack：会新增一个维度，把张量堆叠到新维度上。例：两个[2,3]张量 stack → 变成三维[2,2,3]

In [9]:
t1 = torch.cat([tensor, tensor, tensor], dim=1)
print(t1)

tensor([[1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.]])


**张量的算术运算**

In [ ]:
# 下面计算两个张量的**矩阵乘法**。y1、y2、y3三者计算结果完全相同
# tensor.T 返回张量的转置
y1 = tensor @ tensor.T          # @运算符，矩阵乘法
y2 = tensor.matmul(tensor.T)    # .matmul() 实例方法，矩阵乘法

y3 = torch.rand_like(y1)                     # 创建一个和y1形状一样的随机张量
torch.matmul(tensor, tensor.T, out=y3)        # 矩阵乘法，结果直接输出写入y3（不新建张量，节省内存）

# 下面计算逐元素乘法（哈达玛积），z1、z2、z3三者结果完全相同
z1 = tensor * tensor           # * 逐元素相乘
z2 = tensor.mul(tensor)         # .mul()实例方法，逐元素相乘

z3 = torch.rand_like(tensor)                  # 创建同形状随机张量
torch.mul(tensor, tensor, out=z3)             # 逐元素乘法，结果直接写入z3


tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])

**单元素张量：** 如果你得到只有一个元素的张量（例如把张量全部数值求和聚合得到单个结果），可以用 .item() 将它转为 Python 原生数字。

In [12]:
agg = tensor.sum()        # sum求和，得到形状为 [] 的标量张量，仍然是tensor类型
agg_item = agg.item()     # .item()：取出里面的数字，变成python float
print(agg_item, type(agg_item))

12.0 <class 'float'>


**原地操作：** 把计算结果直接存回操作对象本身，叫做原地操作；函数名字以下划线 _ 结尾代表原地操作。
例子：x.copy_(y)、x.t_()，会直接修改 x 本身

In [13]:
print(f"{tensor} \n")
tensor.add_(5)   # add_() 下划线！原地加5，直接修改tensor自身，不需要赋值
print(tensor)

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]]) 

tensor([[6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.]])


### PART4. PyTorch ↔ NumPy 转换操作

CPU 上的张量与 NumPy 数组共享底层内存，修改其中一个，另一个也会跟着修改。

**张量转 numpy 数组**

In [14]:
t = torch.ones(5)
print(f"t: {t}")
n = t.numpy()   # .numpy() 把tensor转为numpy数组
print(f"n: {n}")

t: tensor([1., 1., 1., 1., 1.])
n: [1. 1. 1. 1. 1.]


在张量中的修改也会同步到numpy数组中：

In [15]:
t.add_(1)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.])
n: [2. 2. 2. 2. 2.]


**numpy 数组转张量**

In [17]:
n = np.ones(5)
t = torch.from_numpy(n)

# 在numpy数组中的修改也会同步到张量中
np.add(n, 1, out=n)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.], dtype=torch.float64)
n: [2. 2. 2. 2. 2.]
